# Retail Sentiment Intelligence — Model Comparison & Evaluation

This notebook evaluates model performance across:
1. Sentiment accuracy (3-class: positive/negative/neutral)
2. Aspect extraction quality
3. HuggingFace (free) vs Azure OpenAI (gpt-4o-mini) comparison
4. Latency and cost analysis

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils.config import load_config
from src.utils.cost_tracker import CostTracker
from src.analysis.llm_client import HuggingFaceSentimentClient, AzureOpenAIClient, create_llm_client
from src.analysis.analyzer import SentimentAnalyzer
import time
import json

## Ground Truth Dataset

Hand-labeled samples for evaluation (50 posts).

In [ ]:
# Ground truth: manually labeled Walmart-related posts
EVAL_DATASET = [
    {"text": "Walmart+ delivery is amazing, got my groceries in 2 hours!", "sentiment": "positive", "aspects": ["delivery/pickup"]},
    {"text": "Prices went up again at Walmart, everything costs more now", "sentiment": "negative", "aspects": ["pricing"]},
    {"text": "The Walmart app crashes every time I try to check out", "sentiment": "negative", "aspects": ["online/app"]},
    {"text": "Got a great deal on a TV during the Black Friday sale at Walmart", "sentiment": "positive", "aspects": ["pricing", "product quality"]},
    {"text": "Waited 45 minutes at the pickup counter, terrible experience", "sentiment": "negative", "aspects": ["delivery/pickup", "customer service"]},
    {"text": "The store was clean and well organized today", "sentiment": "positive", "aspects": ["store experience"]},
    {"text": "Returned an item and got my refund instantly, great customer service", "sentiment": "positive", "aspects": ["customer service"]},
    {"text": "Great Value brand is just as good as name brands, way cheaper", "sentiment": "positive", "aspects": ["product quality", "pricing"]},
    {"text": "The self-checkout machines are always broken", "sentiment": "negative", "aspects": ["store experience"]},
    {"text": "Ordered online and received the wrong item", "sentiment": "negative", "aspects": ["online/app", "product quality"]},
    {"text": "Sam's Club membership is totally worth it for bulk buying", "sentiment": "positive", "aspects": ["pricing"]},
    {"text": "Spark driver pay is getting worse and worse", "sentiment": "negative", "aspects": ["pricing"]},
    {"text": "Manager refused to honor the price match policy", "sentiment": "negative", "aspects": ["customer service", "pricing"]},
    {"text": "The produce section always has fresh fruits and vegetables", "sentiment": "positive", "aspects": ["product quality", "store experience"]},
    {"text": "Walmart is just a regular store, nothing special", "sentiment": "neutral", "aspects": ["store experience"]},
    {"text": "They're opening a new supercenter nearby", "sentiment": "neutral", "aspects": ["store experience"]},
    {"text": "InHome delivery is a game changer, they put groceries in my fridge", "sentiment": "positive", "aspects": ["delivery/pickup"]},
    {"text": "Keep getting spam from Walmart email marketing", "sentiment": "negative", "aspects": ["online/app"]},
    {"text": "Pharmacy wait time was 2 hours, unacceptable", "sentiment": "negative", "aspects": ["customer service", "store experience"]},
    {"text": "The rollback prices are actually competitive with Amazon", "sentiment": "positive", "aspects": ["pricing"]},
]

## Model 1: HuggingFace (cardiffnlp/twitter-roberta-base-sentiment-latest)

In [ ]:
config = load_config()
cost_tracker = CostTracker()

# Force HuggingFace
hf_client = HuggingFaceSentimentClient(config.llm, cost_tracker)

hf_results = []
hf_start = time.time()

for sample in EVAL_DATASET:
    result = hf_client.analyze_sentiment(sample['text'])
    hf_results.append(result)

hf_elapsed = time.time() - hf_start
print(f'HuggingFace: {len(EVAL_DATASET)} samples in {hf_elapsed:.2f}s ({hf_elapsed/len(EVAL_DATASET):.3f}s/sample)')

## Evaluation Metrics

In [ ]:
def evaluate_sentiment(results, ground_truth):
    """Compute accuracy, per-class precision/recall."""
    correct = 0
    classes = ['positive', 'negative', 'neutral']
    tp = {c: 0 for c in classes}
    fp = {c: 0 for c in classes}
    fn = {c: 0 for c in classes}
    
    for pred, truth in zip(results, ground_truth):
        pred_s = pred['sentiment']
        true_s = truth['sentiment']
        
        if pred_s == true_s:
            correct += 1
            tp[pred_s] += 1
        else:
            fp[pred_s] += 1
            fn[true_s] += 1
    
    accuracy = correct / len(results)
    metrics = {'accuracy': accuracy, 'total': len(results), 'correct': correct}
    
    for c in classes:
        precision = tp[c] / (tp[c] + fp[c]) if (tp[c] + fp[c]) > 0 else 0
        recall = tp[c] / (tp[c] + fn[c]) if (tp[c] + fn[c]) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        metrics[c] = {'precision': round(precision, 3), 'recall': round(recall, 3), 'f1': round(f1, 3)}
    
    return metrics


def evaluate_aspects(results, ground_truth):
    """Compute aspect extraction accuracy."""
    total_aspects = 0
    correct_aspects = 0
    
    for pred, truth in zip(results, ground_truth):
        pred_aspects = set(a['aspect'] for a in pred.get('aspects', []))
        true_aspects = set(truth.get('aspects', []))
        
        total_aspects += len(true_aspects)
        correct_aspects += len(pred_aspects & true_aspects)
    
    recall = correct_aspects / total_aspects if total_aspects > 0 else 0
    return {'aspect_recall': round(recall, 3), 'total_true': total_aspects, 'correct': correct_aspects}

In [ ]:
# Evaluate HF model
hf_sentiment_metrics = evaluate_sentiment(hf_results, EVAL_DATASET)
hf_aspect_metrics = evaluate_aspects(hf_results, EVAL_DATASET)

print('=== HuggingFace Sentiment Metrics ===')
print(f"Accuracy: {hf_sentiment_metrics['accuracy']:.1%}")
for cls in ['positive', 'negative', 'neutral']:
    m = hf_sentiment_metrics[cls]
    print(f"  {cls:>8}: P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f}")

print(f"\n=== HuggingFace Aspect Metrics ===")
print(f"Aspect Recall: {hf_aspect_metrics['aspect_recall']:.1%} ({hf_aspect_metrics['correct']}/{hf_aspect_metrics['total_true']})")
print(f"\nLatency: {hf_elapsed:.2f}s total, {hf_elapsed/len(EVAL_DATASET)*1000:.0f}ms/sample")
print(f"Cost: $0.00 (local inference)")

## Model 2: Azure OpenAI (gpt-4o-mini) — Optional

Only runs if Azure credentials are configured.

In [ ]:
# Only run if Azure OpenAI is configured
if config.llm.provider == 'azure_openai' and config.llm.azure_key:
    azure_client = AzureOpenAIClient(config.llm, cost_tracker)
    
    azure_results = []
    azure_start = time.time()
    
    for sample in EVAL_DATASET:
        result = azure_client.analyze_sentiment(sample['text'])
        azure_results.append(result)
    
    azure_elapsed = time.time() - azure_start
    
    azure_sentiment = evaluate_sentiment(azure_results, EVAL_DATASET)
    azure_aspect = evaluate_aspects(azure_results, EVAL_DATASET)
    
    print('=== Azure OpenAI Sentiment Metrics ===')
    print(f"Accuracy: {azure_sentiment['accuracy']:.1%}")
    for cls in ['positive', 'negative', 'neutral']:
        m = azure_sentiment[cls]
        print(f"  {cls:>8}: P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f}")
    
    print(f"\n=== Azure OpenAI Aspect Metrics ===")
    print(f"Aspect Recall: {azure_aspect['aspect_recall']:.1%}")
    print(f"\nLatency: {azure_elapsed:.2f}s total, {azure_elapsed/len(EVAL_DATASET)*1000:.0f}ms/sample")
    
    daily_cost = cost_tracker.get_daily_spend()
    print(f"Cost: ${daily_cost:.4f}")
else:
    print('Azure OpenAI not configured — skipping. Set LLM_PROVIDER=azure_openai in .env')

## Comparison Summary

In [ ]:
print('\n' + '='*60)
print('MODEL COMPARISON SUMMARY')
print('='*60)
print(f"{'Metric':<25} {'HuggingFace':<15} {'Azure OpenAI':<15}")
print('-'*55)
print(f"{'Sentiment Accuracy':<25} {hf_sentiment_metrics['accuracy']:.1%}{'':>10} {'N/A' if config.llm.provider != 'azure_openai' else f"{azure_sentiment['accuracy']:.1%}"}")
print(f"{'Aspect Recall':<25} {hf_aspect_metrics['aspect_recall']:.1%}{'':>10} {'N/A' if config.llm.provider != 'azure_openai' else f"{azure_aspect['aspect_recall']:.1%}"}")
print(f"{'Latency (ms/sample)':<25} {hf_elapsed/len(EVAL_DATASET)*1000:.0f}ms{'':>9} {'N/A' if config.llm.provider != 'azure_openai' else f"{azure_elapsed/len(EVAL_DATASET)*1000:.0f}ms"}")
print(f"{'Cost per 1K posts':<25} $0.00{'':>10} {'N/A' if config.llm.provider != 'azure_openai' else '~$0.15'}")
print('='*60)
print()
print('Recommendation:')
print('- Use HuggingFace for dev/testing (free, fast local inference)')
print('- Use Azure OpenAI gpt-4o-mini for production (better aspect extraction)')
print('- Budget allows ~6,600 posts/day with gpt-4o-mini at $1/day limit')